In [ ]:
# ============================================================
# ALL IMPORTS — gathered here so you only need to run this
# cell once and every cell below can use these libraries.
# ============================================================

import pandas as pd          # DataFrames — our main data structure
import numpy as np           # Numeric math (arrays, linspace, argmax…)
import matplotlib.pyplot as plt  # Base plotting engine
import seaborn as sns        # Higher-level, prettier plots on top of matplotlib
import warnings
warnings.filterwarnings('ignore')  # Silence noisy but harmless sklearn warnings

# --- Sklearn: Model Selection & Validation ---
from sklearn.model_selection import (
    train_test_split,    # Splits data into train/val/test subsets
    StratifiedKFold,     # K-Fold that preserves class ratios in each fold
    cross_val_score,     # Runs CV and returns a score per fold
    learning_curve       # Measures accuracy at different training-set sizes
)

# --- Sklearn: Classifiers ---
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

# --- Sklearn: Evaluation Metrics ---
from sklearn.metrics import (
    accuracy_score,          # (TP + TN) / total  — overall correctness
    classification_report,   # Precision, Recall, F1 per class
    confusion_matrix,        # Table: actual vs predicted counts
    ConfusionMatrixDisplay,  # Plots the confusion matrix as a heatmap
    roc_curve,               # (FPR, TPR) at every threshold
    auc                      # Area under any curve (we use it for ROC)
)

# --- Sklearn: Preprocessing ---
from sklearn.preprocessing import StandardScaler  # Scales features to mean=0, std=1

print("✅ All libraries loaded successfully!")

✅ All libraries loaded successfully!


In [ ]:
#load data 
df = pd.read_csv("d:\SEM 6\DATA SCIENCE\project\datasets\Crime_Data_from_2020_to_2024.csv")

In [ ]:
# filter years to 2022->2024 to reduce size (done only once)
print(f"Dataset before! number of rows: {len(df)}")
#the problem is each value has different formating! ------ 10/18/2020 12:00:00 AM ----- 11/07/2020 00:00
df['DATE OCC'] = pd.to_datetime(df['DATE OCC'] , format = 'mixed') #deals with any format
df = df[df['DATE OCC'].dt.year >= 2022]

# drop useless columns to my case
drop_columns = ['Crm Cd 2','Crm Cd 3','Crm Cd 4','Cross Street','Mocodes']
df = df.drop(columns = drop_columns)

#data size is now reduced
print(f"Dataset reduced! New number of rows: {len(df)}")

Dataset before! number of rows: 1004894
Dataset reduced! New number of rows: 595171


RENAME COLUMNS SO THAT IT'S EASIER FOR ME

In [ ]:
# 1. Automatic clean-up
df.columns = (df.columns
              .str.strip()             
              .str.lower()             
              .str.replace(' ', '_')   
              .str.replace('-', '_'))  

# 2. Rename dictionary
rename_dict = {
    'dr_no': 'incident_id',
    'date_rptd': 'date_reported',
    'date_occ': 'date_occurred',
    'time_occ': 'time_occurred',
    'area': 'area_id',
    'area_name': 'location', # This will now be the ONLY 'location' column
    'rpt_dist_no': 'reporting_district',
    'crm_cd': 'crime_code',
    'crm_cd_desc': 'crime_description',
    'vict_age': 'victim_age',
    'vict_sex': 'victim_sex',
    'vict_descent': 'victim_ethnicity',
    'premis_cd': 'premise_code',
    'premis_desc': 'premise_description',
    'weapon_used_cd': 'weapon_code',
    'weapon_desc': 'weapon_description',
    'status': 'status_code',
    'status_desc': 'status_outcome', 
    'crm_cd_1': 'crime_code_1',
    'lat': 'latitude',
    'lon': 'longitude',
    'part_1_2': 'crime_severity',
    'location' : 'address'
}

# 3. Apply the rename
df = df.rename(columns=rename_dict)

# 4. Feature Engineering
# The 'arrest_made' logic now perfectly matches your 'status_outcome' rename
df['arrest_made'] = df['status_outcome'].apply(lambda x: 'Yes' if isinstance(x, str) and 'Arrest' in x else 'No')

print("\nSuccess! Final columns:\n", df.columns.tolist())


Success! Final columns:
 ['incident_id', 'date_reported', 'date_occurred', 'time_occurred', 'area_id', 'location', 'reporting_district', 'crime_severity', 'crime_code', 'crime_description', 'victim_age', 'victim_sex', 'victim_ethnicity', 'premise_code', 'premise_description', 'weapon_code', 'weapon_description', 'status_code', 'status_outcome', 'crime_code_1', 'address', 'latitude', 'longitude', 'arrest_made']


LET'S START CLEANING

In [ ]:
#drop rows with ages that are less than 0 or 0 since this column is important i need accuracy
df = df[df['victim_age'] > 0]

#clean up victim gender keep (F,M,X'hidden')
valid_gender = ['F','M','X']
df = df [df['victim_sex'].isin(valid_gender )]

#fill empty spaces with usefull info
df['weapon_description'] = df['weapon_description'].fillna('NO WEAPON')

df['victim_ethnicity'] = df['victim_ethnicity'].fillna('unknown')

df['premise_description'] = df['premise_description'].fillna('Unknown')

#lets drop rows where date occured is after date reported 
df['date_occurred'] = pd.to_datetime(df['date_occurred'])
df['date_reported'] = pd.to_datetime(df['date_reported'])

df = df[ df['date_occurred'] <= df['date_reported'] ]

print(f"\nFinal cleaning complete! We have {len(df):,} perfectly clean rows ready for the model.")


Final cleaning complete! We have 424,634 perfectly clean rows ready for the model.


In [ ]:
import os

# --- 1. Check Memory Usage (in your Jupyter Notebook) ---
# This calculates the exact memory your clean dataframe is currently using
memory_bytes = df.memory_usage(deep=True).sum()
memory_mb = memory_bytes / (1024 ** 2) # Convert bytes to Megabytes
print(f"Current DataFrame memory usage: {memory_mb:.2f} MB")

# --- 2. Save the Clean Data and Check File Size (on your computer) ---
# Let's save all your hard work to a brand new file
filename = 'cleaned_crime_data.csv'
df.to_csv(filename, index=False) # index=False stops pandas from writing useless row numbers

# Now we check the actual size of that new file on your hard drive
file_bytes = os.path.getsize(filename)
file_mb = file_bytes / (1024 ** 2)
print(f"New CSV file size on disk: {file_mb:.2f} MB")

Current DataFrame memory usage: 299.67 MB
New CSV file size on disk: 92.22 MB


In [ ]:
# Option 1: Save as a standard CSV 
# (index=False stops pandas from writing useless row numbers)
df.to_csv('cleaned_crime_data.csv', index=False)
print("Saved as CSV!")

# (This heavily compresses the data, making it great for GitHub limits!)
df.to_parquet('cleaned_crime_data.parquet')
print("Saved as Parquet!")

Saved as CSV!
Saved as Parquet!
